# TP02 — OOP
## Complete Exercises 1 to 5

This notebook follows the same format as Exercise 1 and completes the whole TP step by step.

We will build one complete CSV cleaning system using:

1. **OOP Foundation**
2. **Strategy Pattern**
3. **Decorator Pattern**
4. **Factory Pattern**
5. **Template Method Pattern**

Dataset used: `sample_data (2).csv`


# Exercise 1: Build the `CSVReader` Class

### Objective
This exercise practices the basic concepts of Object-Oriented Programming (OOP):

- **Class** — A class is a template for creating objects.
- **Constructor (`__init__`)** — A constructor runs automatically when an object is created.
- **Methods** — A method is a function that belongs to a class.
- **Encapsulation** — Keeping related data and functions together inside one class.

The `CSVReader` class will keep all CSV-reading operations together.


## 1 — Import pandas

We use **pandas** because `pandas.read_csv()` can load a CSV file into a DataFrame.


In [1]:
import pandas as pd

## 2 — Create the `CSVReader` Class

The class contains:

- `__init__(file_path)` → stores the CSV file path.
- `read()` → reads the CSV file using pandas.
- `preview(n)` → prints the first `n` rows.

`self.data` stores the loaded DataFrame so other methods can reuse it.


In [2]:
class CSVReader:
    """A simple reusable class for reading and previewing CSV files."""

    def __init__(self, file_path):
        """Store the CSV file path."""
        self.file_path = file_path
        self.data = None

    def read(self):
        """Read the CSV file and store it in self.data."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows of the loaded CSV file."""
        if self.data is None:
            self.read()

        print(self.data.head(n))

## 3 — Create an Object

The CSV file is in the same folder as this notebook, so we pass its filename to `CSVReader`.


In [3]:
reader = CSVReader("sample_data (2).csv")

## 4 — Read the CSV File

Calling `read()` loads the CSV file and returns a pandas DataFrame.


In [4]:
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 5 — Preview the First 5 Rows

The `preview()` method displays the first rows of the dataset.


In [5]:
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## 6 — Preview a Different Number of Rows

Because `preview(n)` accepts an argument, we can choose how many rows to display.


In [6]:
reader.preview(3)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


## Exercise 1 Result

The `CSVReader` class successfully:

1. Stores the CSV file path.
2. Reads the file using pandas.
3. Stores the DataFrame inside the object.
4. Previews any requested number of rows.


# Exercise 2: Strategy Pattern for Missing Values

### Objective

This exercise introduces the **Strategy Pattern**.

We will create:

- `MissingValueStrategy` → abstract parent class
- `DropMissing` → removes rows with missing values
- `FillMean` → fills numeric missing values with the mean
- `FillMode` → fills missing values with the most frequent value
- `DataCleaner` → applies the selected strategy

The Strategy Pattern lets us change the cleaning method without changing `DataCleaner`.


## 1 — Import Abstract Base Class Tools

`ABC` means **Abstract Base Class**.

`@abstractmethod` defines a method that subclasses must implement.


In [7]:
from abc import ABC, abstractmethod

## 2 — Check Missing Values

`isnull().sum()` counts the missing values in each column.


In [8]:
df.isnull().sum()

id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

## 3 — Create the Abstract Strategy Class

Every missing-value strategy will use the same method name: `handle(df)`.


In [9]:
class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

## 4 — Create `DropMissing`

This strategy removes rows that contain missing values.


In [10]:
class DropMissing(MissingValueStrategy):

    def handle(self, df):
        return df.dropna()

## 5 — Create `FillMean`

This strategy fills missing values in numeric columns using the column mean.


In [11]:
class FillMean(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        numeric_columns = df.select_dtypes(include="number").columns

        for column in numeric_columns:
            df[column] = df[column].fillna(df[column].mean())

        return df

## 6 — Create `FillMode`

The **mode** is the most frequent value in a column.

This strategy can work with both numeric and text columns.


In [12]:
class FillMode(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        for column in df.columns:
            if df[column].isnull().any():
                mode_values = df[column].mode()

                if not mode_values.empty:
                    df[column] = df[column].fillna(mode_values.iloc[0])

        return df

## 7 — Create the `DataCleaner` Class

`DataCleaner` receives a strategy object and uses that strategy to clean the DataFrame.


In [13]:
class DataCleaner:

    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

## 8 — Test `DropMissing`


In [14]:
drop_cleaner = DataCleaner(DropMissing())
df_drop = drop_cleaner.clean(df)

print("Original shape:", df.shape)
print("After DropMissing:", df_drop.shape)
df_drop

Original shape: (7, 7)
After DropMissing: (2, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


## 9 — Test `FillMean`


In [15]:
mean_cleaner = DataCleaner(FillMean())
df_mean = mean_cleaner.clean(df)

print(df_mean.isnull().sum())
df_mean

id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


## 10 — Test `FillMode`


In [16]:
mode_cleaner = DataCleaner(FillMode())
df_mode = mode_cleaner.clean(df)

print(df_mode.isnull().sum())
df_mode

id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


## Exercise 2 Result

We can switch between different missing-value strategies:

```text
DataCleaner
   |
   +-- DropMissing
   +-- FillMean
   +-- FillMode
```

This is the **Strategy Pattern**.


# Exercise 3: Decorators for Logging and Timing

### Objective

This exercise introduces the **Decorator Pattern**.

We will create:

- `@log_action` → prints when a method starts and finishes.
- `@log_time` → measures execution time.

A decorator adds extra behavior to a function without changing its main job.


## 1 — Import `time` and `wraps`

- `time.perf_counter()` measures execution time.
- `wraps` keeps the original function information when we decorate it.


In [17]:
import time
from functools import wraps

## 2 — Create `log_action`

This decorator prints a message before and after a function runs.


In [18]:
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[END] {func.__name__}")
        return result

    return wrapper

## 3 — Create `log_time`

This decorator measures how long a function takes to execute.


In [19]:
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()

        result = func(*args, **kwargs)

        end_time = time.perf_counter()
        execution_time = end_time - start_time

        print(f"[TIME] {func.__name__}: {execution_time:.6f} seconds")
        return result

    return wrapper

## 4 — Apply Decorators to `CSVReader`

We redefine `CSVReader` and add decorators to the `read()` method.


In [20]:
class CSVReader:

    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    @log_action
    @log_time
    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()

        print(self.data.head(n))

## 5 — Test the Decorated `read()` Method


In [21]:
reader = CSVReader("sample_data (2).csv")
df = reader.read()
df

[START] read
[TIME] read: 0.003022 seconds
[END] read


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 6 — Apply Decorators to `DataCleaner`


In [22]:
class DataCleaner:

    def __init__(self, strategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        return self.strategy.handle(df)

## 7 — Test the Decorated `clean()` Method


In [23]:
cleaner = DataCleaner(FillMean())
cleaned_df = cleaner.clean(df)

cleaned_df

[START] clean
[TIME] clean: 0.001087 seconds
[END] clean


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


## Exercise 3 Result

The decorated function now has extra behavior:

```text
START
  ↓
Run original method
  ↓
Show execution time
  ↓
END
```

The main logic of the original method does not need to be rewritten.


# Exercise 4: Factory Pattern for Data Transformations

### Objective

This exercise introduces the **Factory Pattern**.

We will create:

- `DataTransform` → abstract parent class
- `NormalizeColumns`
- `RemoveDuplicates`
- `StandardizeText`
- `TransformFactory`

Instead of manually creating transformation objects, the factory decides which object to create.


## 1 — Create the Abstract `DataTransform` Class

Every transformation must implement `apply(df)`.


In [24]:
class DataTransform(ABC):

    @abstractmethod
    def apply(self, df):
        pass

## 2 — Create `NormalizeColumns`

This transformation:

- removes extra spaces
- converts column names to lowercase
- replaces spaces with underscores


In [25]:
class NormalizeColumns(DataTransform):

    def apply(self, df):
        df = df.copy()

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        return df

## 3 — Create `RemoveDuplicates`

This transformation removes duplicate rows.


In [26]:
class RemoveDuplicates(DataTransform):

    def apply(self, df):
        return df.drop_duplicates()

## 4 — Create `StandardizeText`

This transformation:

- removes extra spaces from text values
- converts text to lowercase


In [27]:
class StandardizeText(DataTransform):

    def apply(self, df):
        df = df.copy()

        text_columns = df.select_dtypes(include="object").columns

        for column in text_columns:
            df[column] = df[column].str.strip().str.lower()

        return df

## 5 — Create `TransformFactory`

The factory receives a string and returns the correct transformation object.


In [28]:
class TransformFactory:

    @staticmethod
    def create_transform(transform_type):

        if transform_type == "normalize":
            return NormalizeColumns()

        elif transform_type == "duplicates":
            return RemoveDuplicates()

        elif transform_type == "text":
            return StandardizeText()

        else:
            raise ValueError(f"Unknown transformation: {transform_type}")

## 6 — Test `NormalizeColumns`


In [29]:
transform = TransformFactory.create_transform("normalize")
df_normalized = transform.apply(df)

print(df_normalized.columns)
df_normalized

Index(['id', 'name', 'age', 'height_cm', 'weight_kg', 'city', 'score'], dtype='object')


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 7 — Test `RemoveDuplicates`


In [30]:
transform = TransformFactory.create_transform("duplicates")
df_no_duplicates = transform.apply(df)

print("Before:", df.shape)
print("After :", df_no_duplicates.shape)
df_no_duplicates

Before: (7, 7)
After : (7, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 8 — Test `StandardizeText`


In [31]:
transform = TransformFactory.create_transform("text")
df_text = transform.apply(df)

df_text

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.0,165.0,68.0,new york,85.0
1,2,bob,NaN,172.0,NaN,los angeles,90.0
2,3,charlie,35.0,168.0,72.0,chicago,NaN
3,4,david,NaN,NaN,80.0,houston,75.0
4,5,eva,27.0,160.0,55.0,new york,88.0
5,6,frank,30.0,175.0,85.0,los angeles,NaN
6,7,grace,NaN,162.0,60.0,chicago,92.0


## 9 — Apply Multiple Transformations


In [32]:
df_transformed = df.copy()

transformations = [
    "normalize",
    "duplicates",
    "text"
]

for transform_name in transformations:
    transform = TransformFactory.create_transform(transform_name)
    df_transformed = transform.apply(df_transformed)

df_transformed

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.0,165.0,68.0,new york,85.0
1,2,bob,NaN,172.0,NaN,los angeles,90.0
2,3,charlie,35.0,168.0,72.0,chicago,NaN
3,4,david,NaN,NaN,80.0,houston,75.0
4,5,eva,27.0,160.0,55.0,new york,88.0
5,6,frank,30.0,175.0,85.0,los angeles,NaN
6,7,grace,NaN,162.0,60.0,chicago,92.0


## Exercise 4 Result

The factory converts a simple string into the correct transformation object:

```text
"normalize"  → NormalizeColumns()
"duplicates" → RemoveDuplicates()
"text"       → StandardizeText()
```

This is the **Factory Pattern**.


# Exercise 5: Full Cleaning Pipeline — Template Method Pattern

### Objective

This exercise combines the previous exercises into one complete cleaning pipeline.

The workflow is:

1. `load()`
2. `clean()`
3. `transform()`
4. `save()`

The base class defines the fixed workflow, and the subclass implements each step.

This is the **Template Method Pattern**.


## 1 — Create the Abstract `DataPipeline`

The `run()` method defines the fixed sequence of the pipeline.


In [33]:
class DataPipeline(ABC):

    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass

## 2 — Create `CSVDataPipeline`

This subclass implements each step for CSV data.


In [34]:
class CSVDataPipeline(DataPipeline):

    def __init__(
        self,
        input_file,
        output_file,
        missing_strategy,
        transformations
    ):
        self.input_file = input_file
        self.output_file = output_file
        self.missing_strategy = missing_strategy
        self.transformations = transformations
        self.df = None

    def load(self):
        print("1. Loading data...")

        reader = CSVReader(self.input_file)
        self.df = reader.read()

        print("Data loaded.")
        print("Shape:", self.df.shape)

    def clean(self):
        print("\n2. Cleaning missing values...")

        cleaner = DataCleaner(self.missing_strategy)
        self.df = cleaner.clean(self.df)

        print("Missing values after cleaning:")
        print(self.df.isnull().sum())

    def transform(self):
        print("\n3. Applying transformations...")

        for transform_name in self.transformations:
            transform = TransformFactory.create_transform(transform_name)
            self.df = transform.apply(self.df)

            print(f"Applied: {transform_name}")

    def save(self):
        print("\n4. Saving cleaned data...")

        self.df.to_csv(self.output_file, index=False)

        print(f"Saved to: {self.output_file}")

## 3 — Create the Pipeline

For this example:

- `FillMean()` handles missing numeric values.
- `"normalize"` cleans column names.
- `"duplicates"` removes duplicate rows.
- `"text"` standardizes text values.


In [35]:
pipeline = CSVDataPipeline(
    input_file="sample_data (2).csv",
    output_file="TP02_cleaned_output.csv",
    missing_strategy=FillMean(),
    transformations=[
        "normalize",
        "duplicates",
        "text"
    ]
)

## 4 — Run the Full Pipeline

Only one command is needed:

```python
pipeline.run()
```

The method automatically follows:

```text
load → clean → transform → save
```


In [36]:
pipeline.run()

1. Loading data...
[START] read
[TIME] read: 0.002535 seconds
[END] read
Data loaded.
Shape: (7, 7)

2. Cleaning missing values...
[START] clean
[TIME] clean: 0.001853 seconds
[END] clean
Missing values after cleaning:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64

3. Applying transformations...
Applied: normalize
Applied: duplicates
Applied: text

4. Saving cleaned data...


Saved to: TP02_cleaned_output.csv


## 5 — Preview the Final Cleaned Data


In [37]:
pipeline.df

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


## 6 — Verify the Saved CSV File


In [38]:
saved_df = pd.read_csv("TP02_cleaned_output.csv")
saved_df

,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


## Exercise 5 Result

The complete workflow is:

```text
CSV file
   ↓
CSVReader
   ↓
Missing-value Strategy
   ↓
Data Transformations
   ↓
Save cleaned CSV
```

The design patterns used across the TP are:

- **Strategy Pattern** → choose how to handle missing values.
- **Decorator Pattern** → add logging and timing.
- **Factory Pattern** → create transformation objects.
- **Template Method Pattern** → define the complete pipeline workflow.


# Final Summary

| Exercise | Main Concept | Main Classes / Functions |
|---|---|---|
| 1 | OOP Foundation | `CSVReader` |
| 2 | Strategy Pattern | `MissingValueStrategy`, `DataCleaner` |
| 3 | Decorator Pattern | `log_action`, `log_time` |
| 4 | Factory Pattern | `DataTransform`, `TransformFactory` |
| 5 | Template Method Pattern | `DataPipeline`, `CSVDataPipeline` |

### Complete TP Flow

```text
Exercise 1
Read CSV
   ↓
Exercise 2
Handle missing values
   ↓
Exercise 3
Add logging and timing
   ↓
Exercise 4
Apply transformations
   ↓
Exercise 5
Combine everything into one pipeline
```
